# Cancer classification app — BreakHis + SIPaKMeD (Kaggle)

**BM3** (Rafay Nisar). A working front-end that predicts the **actual classes** of the two large datasets:

- **Breast histopathology (BreakHis)** -> benign vs malignant
- **Cervical cytology (SIPaKMeD)** -> 5 cell types (Dyskeratotic, Koilocytotic, Metaplastic, Parabasal, Superficial-Intermediate)

It trains a ResNet-50 classifier on each dataset (patient-level split for BreakHis), **saves the models** so re-launch is instant, and serves a Gradio interface with a tab per dataset. Expected accuracy: BreakHis ~87%, SIPaKMeD ~96.5%.

**Setup:** Add Input -> add the BreakHis and SIPaKMeD datasets. Settings -> **GPU T4** + **Internet On**. Then Run All. First run trains the two models (a few minutes each); later runs load the saved models. **Untested by me — paste any error and I'll fix it.**

## 0. Setup + dataset paths

In [1]:
!pip -q install gradio
import os, glob, numpy as np
import torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from PIL import Image
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

# EDIT if your Add-Input folders differ (run the print to check)
BREAKHIS_ROOT = '/kaggle/input/datasets/ambarish/breakhis'
SIPAKMED_ROOT = '/kaggle/input/datasets/marinaeplissiti/sipakmed'
print('input folders:', os.listdir('/kaggle/input'))

device: cuda
input folders: ['datasets']


## 1. Data indexing, model, training (with save/load)

In [2]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
                               transforms.RandomRotation(15), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_tf  = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, paths, labels, tf): self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), int(self.labels[i])

def index_breakhis(root):
    paths, labels, groups = [], [], []
    for p in glob.glob(os.path.join(root, '**', '*.png'), recursive=True):
        lp = p.lower(); labels.append(1 if ('_m_' in os.path.basename(lp) or 'malignant' in lp) else 0)
        paths.append(p)
        parts = os.path.basename(p).replace('.png', '').split('-')
        groups.append(parts[1] + '-' + parts[2] if len(parts) >= 3 else os.path.basename(p))
    return np.array(paths), np.array(labels), np.array(groups), ['benign', 'malignant']

def index_sipakmed(root):
    exts = ('*.bmp', '*.BMP', '*.jpg', '*.png')
    files = [f for e in exts for f in glob.glob(os.path.join(root, '**', e), recursive=True)]
    def cls_of(p):
        for part in p.replace('\\', '/').split('/'):
            if part.lower().startswith('im_'): return part[3:]
        return os.path.basename(os.path.dirname(p))
    raw = [(f, cls_of(f)) for f in files]
    classes = sorted({c for _, c in raw}); cmap = {c: i for i, c in enumerate(classes)}
    return np.array([f for f, _ in raw]), np.array([cmap[c] for _, c in raw]), None, classes

def make_model(n):
    m = torchvision.models.resnet50(weights='IMAGENET1K_V2')
    m.fc = nn.Linear(m.fc.in_features, n)
    return m

def train_or_load(name, paths, labels, groups, classes, epochs=12):
    ckpt = f'/kaggle/working/{name}_model.pt'
    model = make_model(len(classes)).to(dev)
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=dev)); model.eval()
        print(name, 'loaded from', ckpt); return model
    idx = np.arange(len(labels))
    if groups is not None:
        tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(idx, labels, groups))
    else:
        tr, te = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)
    tl = DataLoader(ImgDS(paths[tr], labels[tr], train_tf), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    cw = torch.tensor(len(tr)/(len(classes)*np.bincount(labels[tr], minlength=len(classes))+1e-6), dtype=torch.float32).to(dev)
    crit = nn.CrossEntropyLoss(weight=cw); opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    print(name, 'training', epochs, 'epochs on', len(tr), 'images...')
    for ep in range(epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        print(f'  epoch {ep+1}/{epochs} done', flush=True)
    torch.save(model.state_dict(), ckpt); model.eval(); print(name, 'saved ->', ckpt)
    return model

## 2. Train / load both classifiers

In [3]:
bh_p, bh_y, bh_g, bh_classes = index_breakhis(BREAKHIS_ROOT)
print('BreakHis:', len(bh_p), 'images | classes', bh_classes)
bh_model = train_or_load('breakhis', bh_p, bh_y, bh_g, bh_classes)

sk_p, sk_y, sk_g, sk_classes = index_sipakmed(SIPAKMED_ROOT)
print('SIPaKMeD:', len(sk_p), 'images | classes', sk_classes)
sk_model = train_or_load('sipakmed', sk_p, sk_y, sk_g, sk_classes)

BreakHis: 7909 images | classes ['benign', 'malignant']
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 212MB/s]


breakhis training 12 epochs on 6236 images...
  epoch 1/12 done
  epoch 2/12 done
  epoch 3/12 done
  epoch 4/12 done
  epoch 5/12 done
  epoch 6/12 done
  epoch 7/12 done
  epoch 8/12 done
  epoch 9/12 done
  epoch 10/12 done
  epoch 11/12 done
  epoch 12/12 done
breakhis saved -> /kaggle/working/breakhis_model.pt
SIPaKMeD: 5015 images | classes ['Dyskeratotic', 'Koilocytotic', 'Metaplastic', 'Parabasal', 'Superficial-Intermediate']
sipakmed training 12 epochs on 4012 images...
  epoch 1/12 done
  epoch 2/12 done
  epoch 3/12 done
  epoch 4/12 done
  epoch 5/12 done
  epoch 6/12 done
  epoch 7/12 done
  epoch 8/12 done
  epoch 9/12 done
  epoch 10/12 done
  epoch 11/12 done
  epoch 12/12 done
sipakmed saved -> /kaggle/working/sipakmed_model.pt


## 3. Prediction functions

In [4]:
def predict(model, classes, pil_img):
    if pil_img is None: return {}
    model.eval()
    x = test_tf(pil_img.convert('RGB')).unsqueeze(0).to(dev)
    with torch.no_grad():
        p = torch.softmax(model(x), 1)[0].cpu().numpy()
    return {classes[i]: float(p[i]) for i in range(len(classes))}

def predict_breakhis(img): return predict(bh_model, bh_classes, img)
def predict_sipakmed(img): return predict(sk_model, sk_classes, img)

## 4. Launch the app

In [5]:
import gradio as gr
with gr.Blocks(title='Cancer Classification - BM3') as app:
    gr.Markdown('# Cancer Image Classification (BM3)\nUpload an image on the matching tab to get the predicted class and confidence.')
    with gr.Tab('Breast histopathology (BreakHis)'):
        gr.Markdown('Predicts **benign vs malignant** from a breast-tissue image.')
        bi = gr.Image(type='pil', label='Histopathology image')
        gr.Button('Classify', variant='primary').click(predict_breakhis, bi, gr.Label(num_top_classes=2, label='Prediction'))
    with gr.Tab('Cervical cytology (SIPaKMeD)'):
        gr.Markdown('Predicts the **cell type** (5 classes) from a single-cell Pap-smear image.')
        si = gr.Image(type='pil', label='Cell image')
        gr.Button('Classify', variant='primary').click(predict_sipakmed, si, gr.Label(num_top_classes=5, label='Prediction'))
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4f9ac2392a422cc09b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
## Notes

- To demo: open the share link, drag a test image onto the matching tab, and click Classify. Use an image from the input datasets (e.g. a file under `/kaggle/input/.../malignant/...` for BreakHis, or a `im_Koilocytotic` cell for SIPaKMeD).
- The models are saved to `/kaggle/working`, so re-running loads them instead of retraining. Use **Save Version -> Quick Save** to keep them.
- These two classifiers cover the large public datasets. The **FTIR spectrum** classifier and the **microscopy cell detector** run in the separate Colab app, because their data lives on Google Drive; if all datasets are placed in one environment, the tabs can be merged into a single app.